# 当当网图书榜单数据可视化

使用 Matplotlib 绘制小清新风格图表，配色采用马卡龙色系。

6种分析图表：矩形树图、分组散点图、热力图、词云图、折线图、雷达图

In [ ]:
import configparser
import os

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np
import squarify
from wordcloud import WordCloud
import jieba
import re
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from collections import Counter

PASTEL = ['#7EC8E3','#98D8AA','#F7DC6F','#F0B27A','#C39BD3','#F1948A','#85C1E9','#82E0AA','#F8C471','#D7BDE2']
BG = '#FAFBFC'

for fp in fm.findSystemFonts():
    if 'msyh' in fp.lower() or 'yahei' in fp.lower() or 'simhei' in fp.lower():
        fm.fontManager.addfont(fp)
        prop = fm.FontProperties(fname=fp)
        plt.rcParams['font.family'] = prop.get_name()
        break
plt.rcParams['axes.unicode_minus'] = False

config = configparser.ConfigParser()
config.read(os.path.join(os.getcwd(), 'config.ini'), encoding='utf-8')
DB_HOST = config.get('database', 'host', fallback='localhost')
DB_PORT = config.getint('database', 'port', fallback=3306)
DB_USER = config.get('database', 'username', fallback='root')
DB_PASS = config.get('database', 'password', fallback='')
DB_NAME = config.get('database', 'database', fallback='dangdang_data')

url = f'mysql+pymysql://{DB_USER}:{quote_plus(DB_PASS)}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4'
engine = create_engine(url)
df = pd.read_sql('SELECT * FROM book_rankings', engine)
print(f'共 {len(df)} 条数据')
df.head()

## 1. 分类势力范围（矩形树图）
各图书分类在头部榜单中的"势力范围"，面积代表频次

In [ ]:
cats=df['category'].dropna()
counter=Counter(cats)
top=counter.most_common(12)
labels=[f'{t[0]}\n{t[1]}本' for t in top]
sizes=[t[1] for t in top]
colors=PASTEL[:len(top)]*2
fig,ax=plt.subplots(figsize=(9,6),facecolor=BG)
squarify.plot(sizes=sizes,label=labels,color=colors[:len(sizes)],alpha=0.85,ax=ax,
    text_kwargs={'fontsize':10,'color':'#2c3e50'},edgecolor='white',linewidth=2)
ax.set_title('各分类在榜单中的势力范围',fontsize=14,fontweight='bold',pad=12,color='#2c3e50')
ax.axis('off')
fig.tight_layout()
plt.show()

## 2. 出版社定价区间与排名表现（分组散点图）
横轴=现价，纵轴=排名（倒置），颜色=出版社

In [ ]:
v=df.dropna(subset=['current_price','rank_position','publisher']).copy()
v=v[v['current_price']>0]
pub_counter=Counter(v['publisher'])
top_pubs=[p for p,_ in pub_counter.most_common(6)]
pub_color_map={p:PASTEL[i%len(PASTEL)] for i,p in enumerate(top_pubs)}
fig,ax=plt.subplots(figsize=(9,6),facecolor=BG)
for pub in top_pubs:
    pts=v[v['publisher']==pub]
    ax.scatter(pts['current_price'],pts['rank_position'],c=pub_color_map[pub],
        alpha=0.7,s=50,edgecolors='white',linewidth=0.5,label=pub[:8])
others=v[~v['publisher'].isin(top_pubs)]
if len(others):
    ax.scatter(others['current_price'],others['rank_position'],c='#b0bec5',
        alpha=0.3,s=20,edgecolors='white',linewidth=0.3,label='其他')
ax.invert_yaxis()
ax.legend(fontsize=8,loc='lower right',framealpha=0.8,edgecolor='#e8edf2')
ax.set_facecolor(BG)
ax.set_title('出版社定价区间与排名表现',fontsize=14,fontweight='bold',pad=12,color='#2c3e50')
ax.set_xlabel('当前价格（元）',fontsize=11,color='#7f8c9b')
ax.set_ylabel('排名（越小越好）',fontsize=11,color='#7f8c9b')
ax.tick_params(colors='#7f8c9b',labelsize=9)
ax.spines['top'].set_visible(False);ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#e8edf2');ax.spines['bottom'].set_color('#e8edf2')
ax.grid(axis='y',color='#e8edf2',linestyle='--',linewidth=0.5,alpha=0.7)
fig.tight_layout()
plt.show()

## 3. 价格段与好评度热度关联（热力图）
寻找"黄金甜蜜点"——便宜且口碑好的书最容易上榜

In [ ]:
v=df.dropna(subset=['current_price','rating']).copy()
v=v[(v['current_price']>0)&(v['rating']>0)]
def price_bin(p):
    if p<30:return '0-30元'
    elif p<60:return '30-60元'
    elif p<100:return '60-100元'
    else:return '100元+'
def rating_bin(r):
    if r<85:return '85%以下'
    elif r<90:return '85-90%'
    elif r<95:return '90-95%'
    else:return '95-100%'
price_order=['0-30元','30-60元','60-100元','100元+']
rating_order=['85%以下','85-90%','90-95%','95-100%']
matrix=np.zeros((len(rating_order),len(price_order)))
for _,row in v.iterrows():
    pb,rb=price_bin(row['current_price']),rating_bin(row['rating'])
    if pb in price_order and rb in rating_order:
        matrix[rating_order.index(rb)][price_order.index(pb)]+=1
fig,ax=plt.subplots(figsize=(8,5.5),facecolor=BG)
im=ax.imshow(matrix,cmap='YlOrRd',aspect='auto',interpolation='nearest')
ax.set_xticks(range(len(price_order)));ax.set_xticklabels(price_order,fontsize=10,color='#2c3e50')
ax.set_yticks(range(len(rating_order)));ax.set_yticklabels(rating_order,fontsize=10,color='#2c3e50')
for i in range(len(rating_order)):
    for j in range(len(price_order)):
        val=int(matrix[i][j])
        color='white' if val>matrix.max()*0.6 else '#2c3e50'
        ax.text(j,i,str(val),ha='center',va='center',fontsize=12,fontweight='bold',color=color)
cbar=fig.colorbar(im,ax=ax,shrink=0.8)
cbar.set_label('图书数量',fontsize=10,color='#7f8c9b')
cbar.ax.tick_params(colors='#7f8c9b',labelsize=8)
ax.set_facecolor(BG)
ax.set_title('价格段与好评度热度关联（寻找黄金甜蜜点）',fontsize=14,fontweight='bold',pad=12,color='#2c3e50')
ax.set_xlabel('价格区间',fontsize=11,color='#7f8c9b')
ax.set_ylabel('好评度区间',fontsize=11,color='#7f8c9b')
ax.tick_params(colors='#7f8c9b',labelsize=9)
fig.tight_layout()
plt.show()

## 4. 畅销书标题流量密码（词云图）
书名关键词频率分析，发现选题趋势

In [ ]:
titles=df['book_title'].dropna().tolist()
all_text=' '.join(titles)
stopwords={'的','了','与','和','及','或','在','是','有','为','中','等','上','下','不','一','个','这','那',
    '我','你','他','她','它','们','着','过','地','得','会','能','要','就','也','都','而','及','其','之',
    '以','于','从','到','被','把','让','给','向','对','当','将','所','如','可','但','又','很','最','更','还','再','才','已'}
words=jieba.cut(all_text)
filtered=[w for w in words if len(w)>=2 and w not in stopwords and not re.match(r'^[\d\W]+$',w)]
text=' '.join(filtered)
font_path=None
for fp in fm.findSystemFonts():
    if 'msyh' in fp.lower() or 'yahei' in fp.lower() or 'simhei' in fp.lower():
        font_path=fp;break
wc=WordCloud(font_path=font_path,width=900,height=500,background_color=BG,
    max_words=80,max_font_size=120,colormap='Set2',contour_width=0,margin=10)
wc.generate(text)
fig,ax=plt.subplots(figsize=(9,5.5),facecolor=BG)
ax.imshow(wc,interpolation='bilinear')
ax.set_title('畅销书标题流量密码',fontsize=14,fontweight='bold',pad=12,color='#2c3e50')
ax.axis('off')
fig.tight_layout()
plt.show()

## 5. 出版时间趋势分析（折线图）
观察新书的"半衰期"或"爆发力"

In [ ]:
v=df.dropna(subset=['publish_date','rank_position']).copy()
month_data={}
for _,row in v.iterrows():
    m=re.match(r'(\d{4})[-./年](\d{1,2})',str(row['publish_date']))
    if m:
        key=f'{m.group(1)}-{int(m.group(2)):02d}'
        month_data.setdefault(key,[]).append(row['rank_position'])
if month_data:
    months=sorted(month_data.keys())
    avg_ranks=[sum(month_data[m])/len(month_data[m]) for m in months]
    counts=[len(month_data[m]) for m in months]
    fig,ax1=plt.subplots(figsize=(9,5),facecolor=BG)
    ax1.plot(months,avg_ranks,color=PASTEL[5],marker='o',linewidth=2,markersize=6,label='平均排名')
    ax1.invert_yaxis()
    ax1.set_facecolor(BG)
    ax1.set_title('出版时间趋势分析（半衰期与爆发力）',fontsize=14,fontweight='bold',pad=12,color='#2c3e50')
    ax1.set_xlabel('出版月份',fontsize=11,color='#7f8c9b')
    ax1.set_ylabel('平均排名（越小越好）',fontsize=11,color='#7f8c9b')
    ax1.tick_params(colors='#7f8c9b',labelsize=9)
    ax1.spines['top'].set_visible(False);ax1.spines['right'].set_visible(False)
    ax1.spines['left'].set_color('#e8edf2');ax1.spines['bottom'].set_color('#e8edf2')
    ax1.grid(axis='y',color='#e8edf2',linestyle='--',linewidth=0.5,alpha=0.7)
    ax2=ax1.twinx()
    ax2.bar(months,counts,alpha=0.25,color=PASTEL[0],label='上榜数量',width=0.6)
    ax2.set_ylabel('上榜数量',fontsize=11,color='#7f8c9b')
    ax2.tick_params(colors='#7f8c9b',labelsize=9)
    ax2.spines['top'].set_visible(False);ax2.spines['right'].set_color('#e8edf2')
    lines1,labels1=ax1.get_legend_handles_labels()
    lines2,labels2=ax2.get_legend_handles_labels()
    ax1.legend(lines1+lines2,labels1+labels2,loc='upper right',fontsize=9,framealpha=0.8)
    plt.setp(ax1.get_xticklabels(),rotation=45,ha='right',fontsize=8)
    fig.tight_layout()
    plt.show()
else:
    print('无有效出版时间数据')

## 6. 头部出版社综合实力画像（雷达图）
对比巨头的竞争优势：平均排名、好评度、价格竞争力

In [ ]:
pub_data={}
for _,row in df.iterrows():
    pub=row.get('publisher','')
    if not pub or pd.isna(pub):continue
    pub_data.setdefault(pub,{'ranks':[],'ratings':[],'discounts':[]})
    if not pd.isna(row.get('rank_position')):pub_data[pub]['ranks'].append(row['rank_position'])
    if not pd.isna(row.get('rating')) and row['rating']>0:pub_data[pub]['ratings'].append(row['rating'])
    if not pd.isna(row.get('discount')) and row['discount']>0:pub_data[pub]['discounts'].append(row['discount'])
pub_scores=[]
for pub,d in pub_data.items():
    if len(d['ranks'])<2:continue
    avg_rank=sum(d['ranks'])/len(d['ranks'])
    avg_rating=sum(d['ratings'])/len(d['ratings']) if d['ratings'] else 0
    avg_discount=sum(d['discounts'])/len(d['discounts']) if d['discounts'] else 5
    pub_scores.append((pub,avg_rank,avg_rating,avg_discount,len(d['ranks'])))
if len(pub_scores)>=2:
    pub_scores.sort(key=lambda x:x[4],reverse=True)
    top3=pub_scores[:3]
    categories=['平均排名\n(越小越好)','好评度\n(越高越好)','价格竞争力\n(折扣越低越好)']
    N=len(categories)
    angles=np.linspace(0,2*np.pi,N,endpoint=False).tolist()
    angles+=angles[:1]
    fig,ax=plt.subplots(figsize=(7,7),facecolor=BG,subplot_kw=dict(polar=True))
    max_rank=max(p[1] for p in top3)*1.2
    for idx,(pub,avg_rank,avg_rating,avg_discount,count) in enumerate(top3):
        rank_norm=(max_rank-avg_rank)/max_rank*100 if max_rank>0 else 50
        rating_norm=avg_rating/100*100
        discount_norm=(10-avg_discount)/10*100
        values=[rank_norm,rating_norm,discount_norm]
        values+=values[:1]
        color=PASTEL[idx*3%len(PASTEL)]
        ax.plot(angles,values,'o-',linewidth=2,label=f'{pub[:8]}({count}本)',color=color,markersize=5)
        ax.fill(angles,values,alpha=0.15,color=color)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories,fontsize=10,color='#2c3e50')
    ax.set_ylim(0,100)
    ax.set_yticks([25,50,75,100])
    ax.set_yticklabels(['25','50','75','100'],fontsize=8,color='#7f8c9b')
    ax.legend(loc='upper right',bbox_to_anchor=(1.3,1.1),fontsize=9,framealpha=0.8)
    ax.set_title('头部出版社综合实力画像',fontsize=14,fontweight='bold',pad=20,color='#2c3e50')
    fig.tight_layout()
    plt.show()
else:
    print('数据不足，至少需要2个出版社各2本书')